In [ ]:
# %% Deep learning - Section 20.189
#    CNN project 1: import and classify cifar10
#    1) Import the CIFAR10 dataset, apply transformations, and create a train,
#       dev, and test set
#    2) Make sure the data are 3x32x32
#    3) You are free to select the architecture and the metaparameters
#    4) Categorise the images with a CNN, don't stress about the accuracy

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Import data

cifar10_data = torchvision.datasets.CIFAR10(root='cifar10',download=True)
print(cifar10_data)


In [ ]:
# %% Inspect dataset

# Shape of the dataset (num of images, [size, size], RGB channels)
print("Data shape :")
print(cifar10_data.data.shape), print()

# Unique categories
print("Categories :")
print(cifar10_data.classes), print()

# .targets is a list of targets converted to integers
print("First few targets (int) :")
print(cifar10_data.targets[:10]), print()

print("Target length :")
print(len(cifar10_data.targets))


In [ ]:
# %% Plotting

# Inspect a few random images
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(5,5,figsize=(phi*9,9))

for ax in axs.flatten():

    randidx = np.random.choice(len(cifar10_data.targets))

    # Get image and label
    pic   = cifar10_data.data[randidx,:,:,:]
    label = cifar10_data.classes[cifar10_data.targets[randidx]]

    ax.imshow(pic)
    ax.text(16,0,label,ha='center',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure1_cnn_project_1.png')
plt.show()
files.download('figure1_cnn_project_1.png')


In [6]:
# %% Split data

# Define train, dev, test splits (80/10/10)
indices = np.arange(len(cifar10_data.targets))
np.random.shuffle(indices)

train_boundary = int(0.8 * len(indices))
dev_boundary   = int(0.9 * len(indices))

train_idx = indices[:train_boundary]
dev_idx   = indices[train_boundary:dev_boundary]
test_idx  = indices[dev_boundary:]


In [ ]:
# %% Temporary transform to compute train mean/std for normalisation

# Temporarily convert images to tensors only
cifar10_data.transform = T.ToTensor()

train_subset = Subset(cifar10_data,train_idx)
train_loader = DataLoader(train_subset,batch_size=5000,shuffle=False)

# Accumulate images to compute mean/std from train dataset only
all_images = []

for imgs,_ in train_loader:
    all_images.append(imgs)

# Shape [N_images,chans,H,W]
all_images = torch.cat(all_images,dim=0)

mean = all_images.mean(dim=[0,2,3])
std  = all_images.std(dim=[0,2,3])

print("Train mean:",mean)
print("Train std:",std)


In [15]:
# %% Transformation to tensor and data augmentation

# Define transforms
train_transform = T.Compose([ T.ToPILImage(),
                              T.RandomRotation((-15,15)),
                              T.RandomAffine(degrees=0,translate=(0.1,0.1),shear=10),
                              T.ToTensor(),
                              T.Normalize(mean,std) ])

test_transform = T.Compose([ T.ToPILImage(),
                             T.ToTensor(),
                             T.Normalize(mean, std) ])


In [9]:
# %% Custom dataset

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self,dataset,indices,transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __getitem__(self,idx):
        x,y = self.dataset[self.indices[idx]]
        if self.transform:
            x = self.transform(x)
        return x,y

    def __len__(self):
        return len(self.indices)


In [14]:
# %% Define sets and apply transforms

# Split data, convert to PyTorch datasets, and apply transforms
train_dataset = CustomDataset(cifar10_data, train_idx, train_transform)
dev_dataset   = CustomDataset(cifar10_data, dev_idx,   test_transform)
test_dataset  = CustomDataset(cifar10_data, test_idx,  test_transform)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,drop_last=True)
dev_loader   = DataLoader(dev_dataset,batch_size=len(dev_dataset))
test_loader  = DataLoader(test_dataset,batch_size=len(test_dataset))


In [ ]:
# %% Check DataLoaders

images, labels = next(iter(train_loader))
print("Train batch (img;labels):", images.shape, labels.shape)

images, labels = next(iter(dev_loader))
print("Dev set (img;labels):", images.shape, labels.shape)

images, labels = next(iter(test_loader))
print("Test set (img;labels):",images.shape,labels.shape)


In [ ]:
# Plotting

# Unnormalize images for display (avoid modifying original)
def unnormalize(img,mean,std):
    img = img.clone()
    for c in range(3):
        img[c] = img[c]*std[c]+mean[c]
    return img

# Get one batch
X, y = next(iter(train_loader))  # X: [B,3,32,32]

# Plot
phi = (1 + np.sqrt(5)) / 2
fig, axs = plt.subplots(3, 7, figsize=(phi*6,6))

for i, ax in enumerate(axs.flatten()):

    whichpic = np.random.randint(X.shape[0])
    img      = X[whichpic]
    label    = y[whichpic].item()

    # Unnormalize for display
    img_disp = unnormalize(img, mean, std)

    # Convert to from CHW to HWC and numpy, for matplotlib
    img_disp = img_disp.permute(1,2,0).numpy()

    # Clip values to [0,1]
    img_disp = np.clip(img_disp,0,1)

    ax.imshow(img_disp)
    ax.text(16,0,cifar10_data.classes[label],ha='center',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()
plt.savefig('figure2_cnn_project_1.png')
plt.show()
files.download('figure2_cnn_project_1.png')


In [16]:
# %% Function to generate the model

def gen_model():

    class cifar10_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Use blocks of Conv-Conv-Pool (VGG style)
            # Convolution block 1
            self.conv1_1 = nn.Conv2d(3,32,kernel_size=3,padding=1)
            self.conv1_2 = nn.Conv2d(32,32,kernel_size=3,padding=1)

            # Batch normalisation and dropout for convolution
            self.bnorm1 = nn.BatchNorm2d(32)
            self.drop1  = nn.Dropout2d(p=0.15)

            # Convolution block 2
            self.conv2_1 = nn.Conv2d(32,64,kernel_size=3,padding=1)
            self.conv2_2 = nn.Conv2d(64,64,kernel_size=3,padding=1)

            # Batch normalisation and dropout for convolution
            self.bnorm2 = nn.BatchNorm2d(64)
            self.drop2  = nn.Dropout2d(p=0.15)

            # Fully connected layer with batch normalisation
            self.fc1       = nn.Linear(8*8*64,50)
            self.bnorm_fc1 = nn.BatchNorm1d(50)
            self.drop_fc1  = nn.Dropout(p=0.5)

            # Output layer (10 categories)
            self.output = nn.Linear(50,10)

        def forward(self,x):

            # MaxPool and ReLu on convolution block 1 (pool after passing to
            # activation function)
            x = F.leaky_relu(self.conv1_1(x))
            x = F.leaky_relu(self.conv1_2(x))
            x = F.max_pool2d(x,2)
            x = self.bnorm1(x)
            x = self.drop1(x)

            # MaxPool and ReLu on convolution block 2
            x = F.leaky_relu(self.conv2_1(x))
            x = F.leaky_relu(self.conv2_2(x))
            x = F.max_pool2d(x,2)
            x = self.bnorm2(x)
            x = self.drop2(x)

            # Vectorise for linear layer
            n_units = x.shape.numel() / x.shape[0]
            x       = x.view(-1,int(n_units))

            # Linear and output layers
            x = F.leaky_relu(self.fc1(x))
            x = self.bnorm_fc1(x)

            x = self.output(x)

            return x

    # Create model instance
    CNN = cifar10_CNN()

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [20]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model()

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    dev_losses   = []
    train_acc    = []
    dev_acc      = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and error rate (1-accuracy) from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append( loss.item() )
            batch_acc.append( torch.mean( (torch.argmax(yHat,axis=1) == y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Dev accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(dev_loader))

            # Ship to GPU (y for loss calculation)
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            yHat = yHat.cpu()
            y    = y.cpu()

            dev_acc.append( 100*torch.mean( (torch.argmax(yHat,axis=1) == y).float() ).item() )
            dev_losses.append(loss.item())

        CNN.train()

    return train_acc,dev_acc,train_losses,dev_losses,CNN


In [ ]:
# %% Fit model

# Takes ~8 mins on GPU
train_acc,dev_acc,train_losses,dev_losses,CNN = train_model()


In [ ]:
# %% Pass test through model

# Test accuracy
CNN.eval()
with torch.no_grad():
    X_test,y_test = next(iter(test_loader))
    X_test,y_test = X_test.to(device),y_test.to(device)
    yHat          = CNN(X_test)
    preds         = torch.argmax(yHat,axis=1)
    test_accuracy = 100 * (preds == y_test).sum().item() / y_test.size(0)

print(f"Test accuracy: {test_accuracy:.2f}%")


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig, ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_losses,'s-',alpha=0.7,label='Train')
ax[0].plot(dev_losses,'o-',alpha=0.7,label='Dev')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc,'s-',alpha=0.7,label=f'Train: {train_acc[-1]:.2f}%')
ax[1].plot(dev_acc,'o-',alpha=0.7,label=f'Dev: {dev_acc[-1]:.2f}%')
epoch_idx = len(train_acc) - 1
ax[1].plot(epoch_idx, test_accuracy, marker='^', color='r', markersize=10,alpha=0.7,label=f'Test: {test_accuracy:.2f}%')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model test accuracy: {test_accuracy:.2f}%')
ax[1].legend()

plt.tight_layout()
plt.savefig('figure3_cnn_project_1.png')
plt.show()
files.download('figure3_cnn_project_1.png')


In [ ]:
# %% Plotting

# Get data from test dataloader
X,y  = next(iter(test_loader))
X,y  = X.to(device), y.to(device)
yHat = CNN(X)

# Pick random examples
rand_id = np.random.choice(len(y),size=21,replace=False)

# Plot
phi = (1 + np.sqrt(5)) / 2
fig, axs = plt.subplots(3,7,figsize=(1.5*phi*6,6))

for i, ax in enumerate(axs.flatten()):
    idx = rand_id[i]

    # Unnormalize image and convert to HWC for matplotlib
    img = unnormalize(X[idx],mean,std)
    img = img.permute(1,2,0).cpu().numpy()
    img = np.clip(img,0,1)

    # True and predicted labels
    true_label = cifar10_data.classes[y[idx].item()]
    pred_label = cifar10_data.classes[torch.argmax(yHat[idx]).item()]

    # Set background color: yellow if correct, red if wrong
    bgcolor = 'yellow' if true_label == pred_label else 'red'

    ax.imshow(img)
    ax.text(16,0,f'{true_label}\n{pred_label}',ha='center',fontweight='bold',color='k',backgroundcolor=bgcolor)
    ax.axis('off')

plt.tight_layout()
plt.savefig('figure4_cnn_project_1.png')
plt.show()
files.download('figure4_cnn_project_1.png')


In [ ]:
# %% Plotting

# Get predictions from the test set
CNN.eval()
with torch.no_grad():
    X_test,y_test = next(iter(test_loader))
    X_test,y_test = X_test.to(device), y_test.to(device)
    yHat_test = CNN(X_test)
    preds = torch.argmax(yHat_test,axis=1)

# Normalised confusion matrix
C = skm.confusion_matrix(y_test.cpu(),preds.cpu(),normalize='true')

# Plot
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(7*phi,7))
plt.imshow(C,cmap='Blues',vmin=0,vmax=1)

labels = cifar10_data.classes
plt.xticks(range(len(labels)),labels,rotation=45)
plt.yticks(range(len(labels)),labels)

plt.title('Confusion matrix - CIFAR-10 test set\n(full range)')
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.colorbar(label='Proportion per true class')
plt.tight_layout()

plt.savefig('figure5_cnn_project_1.png')
plt.show()
files.download('figure5_cnn_project_1.png')
